In [1]:
import os
import sys
import numpy as np
from torch.utils.data import Dataset
import torch
from finetune_model_dunwei import ft_1lead_ECGFounder
import pandas as pd
from tqdm import tqdm
import gc

In [2]:
data_dir = r'D:\M143020071\MACE and MI\raw_data_result\iSKNA_signal\ch1\sr500_0.5_50_MI_win10s_step2s_2-7m'
# 特徵儲存的根目錄
save_dir = r'D:\M143020071\MACE and MI\xgboost_results\ECG_Founder_feature'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"Created directory: {save_dir}")
else:
    print(f"Directory already exists: {save_dir}")

Directory already exists: D:\M143020071\MACE and MI\xgboost_results\ECG_Founder_feature


In [3]:
gpu_id = 0
device = torch.device('cuda:{}'.format(gpu_id) if torch.cuda.is_available() else 'cpu')
n_classes = 150
pth = r'F:\M143020071\MI\program\ECG_Founder\checkpoint\1_lead_ECGFounder.pth' # model save checkpoint path
model = ft_1lead_ECGFounder(device, pth, n_classes,linear_prob=False)

In [4]:
signal_list = []
label_list = []
ID_list = []

for root, dirs, files in os.walk(data_dir):
    for filename in files:
        if not filename.endswith('.npy'):
            continue
            
        file_path = os.path.join(root, filename) # 使用 root 確保路徑正確
        data = np.load(file_path)
        signal = data[:, 1:]  
        label = data[:, 0]   
        ID = filename.split('.')[0]
        signal_list.append(signal)
        label_list.append(label)
        ID_list.extend([ID] * len(label))
signals = np.concatenate(signal_list, axis=0)  # shape (N, 5000)
labels = np.concatenate(label_list, axis=0)    # shape (N,)
IDs = np.array(ID_list)  # Convert list of IDs to numpy array
del signal_list, label_list, ID_list  # 刪除舊的列表引用
gc.collect()
print(f"Loaded signals shape: {signals.shape}")
print(f"Loaded labels shape: {labels.shape}")
print(f"Loaded IDs shape: {IDs.shape}")


Loaded signals shape: (197170, 5000)
Loaded labels shape: (197170,)
Loaded IDs shape: (197170,)


In [5]:
gpu_id = 0
device = torch.device('cuda:{}'.format(gpu_id) if torch.cuda.is_available() else 'cpu')
n_classes = 150
pth = r'F:\M143020071\MI\program\ECG_Founder\checkpoint\1_lead_ECGFounder.pth' # model save checkpoint path
model = ft_1lead_ECGFounder(device, pth, n_classes,linear_prob=False)

In [6]:
# batch size for feature extraction
feature_list = []
deep_feature_list = []
batch_size = 1024
for i in range(0, signals.shape[0], batch_size):
    if i + batch_size > signals.shape[0]:
        batch_size = signals.shape[0] - i
    batch_signals = signals[i:i+batch_size]
    batch_signals_tensor = torch.tensor(batch_signals, dtype=torch.float32).to(device).unsqueeze(1)  # shape (batch_size, 1, 5000)
    with torch.no_grad():
        features, deep_features = model(batch_signals_tensor)
        features = features.cpu().numpy()
        deep_features = deep_features.cpu().numpy()
        feature_list.append(features)
        deep_feature_list.append(deep_features)
    del batch_signals_tensor, features, deep_features

torch.cuda.empty_cache()
features = np.concatenate(feature_list, axis=0)  # shape (N, feature_dim)
deep_features = np.concatenate(deep_feature_list, axis=0)  # shape (N, deep_feature_dim)
del feature_list, deep_feature_list
gc.collect()
features_with_id_label = np.concatenate((IDs.reshape(-1, 1), labels.reshape(-1, 1), features), axis=1)  # shape (N, 1 + 1 + feature_dim)
deep_features_with_id_label = np.concatenate((IDs.reshape(-1, 1), labels.reshape(-1, 1), deep_features), axis=1)  # shape (N, 1 + 1 + deep_feature_dim)

print(f"Extracted features shape: {features_with_id_label.shape}")
print(f"Extracted deep features shape: {deep_features_with_id_label.shape}")
np.save(os.path.join(save_dir, 'ECG_Founder_features.npy'), features_with_id_label)
np.save(os.path.join(save_dir, 'ECG_Founder_deep_features.npy'), deep_features_with_id_label)


Extracted features shape: (197170, 152)
Extracted deep features shape: (197170, 1026)
